In [1]:
import numpy as np
import jax
import jaxlib
import jax.numpy as jnp
import flax
import flax.linen as nn
import optax
from typing import Tuple, Callable, Any, Dict, Optional
import numpy.typing as npt
import copy
import pathlib
import matplotlib.pyplot as plt
import time
import json
import ast
import netket as nk
import os
import glob
import sys

sys.path.append("/home/ihuarte/Escritorio/Ivan/NNs")

# os.chdir("/home/ihuarte/Escritorio/Ivan/NN/")

from VA_project.model.model import J1J2Square
from VA_project.engine.runners import Runner

# from NN_utils import load_vstate
# from correlations import correlations_vstate

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")
print(jax.devices())

# from NNs.NN_module.ST_utils import compare_params, masked_optimizer
from frozendict import deepfreeze
from NN_module.ST_utils import print_tree

[CpuDevice(id=0)]


In [2]:
configurations = [
    "/home/ihuarte/Escritorio/Ivan/NNs/config_Hydra.json",
    "/home/ihuarte/Escritorio/Ivan/NNs/config_Hydra_NN.json",
]
with open(configurations[0], "r") as f:
    config_hydra = json.load(f)
with open(configurations[1], "r") as f:
    config_nn = json.load(f)

config = {**config_hydra, **config_nn}

In [3]:
storage = config["storage"]
symm_wrappers = config["symm_wrappers"]
arch_evolution = config[config["selection"]]

In [64]:
submodules_dict = {
    "SplitTraining": ["modulus", "phase"],
    "Sequential": "Seq",
    "Transversal": "Trans",
}


def get_submodules(father, n_mod):
    if father == "Sequential":
        submodules = [f"Seq_{i}" for i in range(n_mod - 1)] + ["End"]
    elif father == "Transversal":
        submodules = [f"Trans_{i}" for i in range(n_mod)]
    elif father == "SplitTraining":
        submodules = ["modulus", "phase"]
    return submodules


def config_from_template(template, storage, symm_wrappers=None, father=""):
    print(f"father: {father}")
    print(f"template: {template}")

    config = {}
    if not father:
        print("1\n")
        for k, v in template.items():
            return config_from_template(v, storage, symm_wrappers=symm_wrappers, father=k)

    elif father in submodules_dict.keys():
        print("2\n")
        n_mod = len([1 for _, v in template.items() if isinstance(v, dict)])
        submodules = get_submodules(father, n_mod)
        config["name"] = father
        config["setup"] = {}
        for i, (k, v) in enumerate(template.items()):
            if isinstance(v, dict):
                config['setup'].update(**{submodules[i] : config_from_template(v, storage, father=k)})
            else:
                config['setup'].update(**{k:v})

    else:
        print(bool(template))
        print("3\n") 
        if isinstance(template, dict):
            

            if template:
                for k, v in template.items():
                    config["name"] = k
                    config["setup"] = config_from_template(v, storage, father=k)

            else:
                config[father] = storage[father]

        else:
            config[father] = template

    if symm_wrappers is not None:
        config.update(**symm_wrappers)

    # print(config,'\n')


    return config


configuration = config_from_template(arch_evolution["model_0"], storage, symm_wrappers)

father: 
template: {'SplitTraining': {'Transversal': {'CNN': {}, 'CvT': {}, 'operaton': 'sum'}, 'Sequential': {'CNNSzabo': {}, 'Sequential': {'CvT': {}, 'ViT2D': {}, 'MLP': {}}}}}
1

father: SplitTraining
template: {'Transversal': {'CNN': {}, 'CvT': {}, 'operaton': 'sum'}, 'Sequential': {'CNNSzabo': {}, 'Sequential': {'CvT': {}, 'ViT2D': {}, 'MLP': {}}}}
2

father: Transversal
template: {'CNN': {}, 'CvT': {}, 'operaton': 'sum'}
2

father: CNN
template: {}
False
3

father: CvT
template: {}
False
3

father: Sequential
template: {'CNNSzabo': {}, 'Sequential': {'CvT': {}, 'ViT2D': {}, 'MLP': {}}}
2

father: CNNSzabo
template: {}
False
3

father: Sequential
template: {'CvT': {}, 'ViT2D': {}, 'MLP': {}}
2

father: CvT
template: {}
False
3

father: ViT2D
template: {}
False
3

father: MLP
template: {}
False
3



In [65]:
print_tree(configuration, values=True)

name: SplitTraining
setup
  modulus
    name: Transversal
    setup
      Trans_0
        CNN
          channels: [32, 16, 8, 4]
          strides: [[1, 1], [1, 1], [1, 1], [1, 1]]
          kernel: [3, 3]
          use_pooling: False
          pooling_strides: [[1, 1], [1, 1], [1, 1], [1, 1]]
          final_architecture: [1]
      Trans_1
        CvT
          n_CP_blocks: [2]
          channels: [32]
          attn_heads: [4]
          kernel: [3, 3]
          final_architecture: [1]
      operaton: sum
  phase
    name: Sequential
    setup
      Seq_0
        CNNSzabo
          channels: 40
          use_bias: True
      End
        name: Sequential
        setup
          Seq_0
            CvT
              n_CP_blocks: [2]
              channels: [32]
              attn_heads: [4]
              kernel: [3, 3]
              final_architecture: [1]
          Seq_1
            ViT2D
              token_size: [2, 1]
              embedding_d: 32
              n_heads: 2
            

In [62]:
configuration

{'name': 'SplitTraining',
 'setup': {'modulus': {'name': 'Transversal',
   'setup': {'Trans_0': {'CNN': {'channels': [32, 16, 8, 4],
      'strides': [[1, 1], [1, 1], [1, 1], [1, 1]],
      'kernel': [3, 3],
      'use_pooling': False,
      'pooling_strides': [[1, 1], [1, 1], [1, 1], [1, 1]],
      'final_architecture': [1]}},
    'Trans_1': {'CvT': {'n_CP_blocks': [2],
      'channels': [32],
      'attn_heads': [4],
      'kernel': [3, 3],
      'final_architecture': [1]}}},
   'operaton': 'sum'},
  'phase': {'name': 'Sequential',
   'setup': {'Seq_0': {'CNNSzabo': {'channels': 40, 'use_bias': True}},
    'End': {'name': 'Sequential',
     'setup': {'Seq_0': {'CvT': {'n_CP_blocks': [2],
        'channels': [32],
        'attn_heads': [4],
        'kernel': [3, 3],
        'final_architecture': [1]}},
      'Seq_1': {'ViT2D': {'token_size': [2, 1],
        'embedding_d': 32,
        'n_heads': 2,
        'n_blocks': 2,
        'n_ffn_layers': 1,
        'final_architecture': [5]}},
 

In [ ]:
from NN_module.schedule.schedule import Schedule
import optax

setup = {
    "print_arch": True,
    "learning_rate": {
        "epochs_struct": [[[50], [100], [200]]],
        "modes_struct": [[[["A"]], [["1", "110"]], [["10000"]]]],
        "lr_struct": [
            [[[0.01, 0.05]], [["lin(0.001, 0.1)", 0.3]], [["exp(0.1, 0.001)"]]]
        ],
        "repeat": [],
        "rescale": 1.0,
    },
    "architecture": {},
    "sampler": {},
}


sch = Schedule(setup, vstate.parameters)
sch.total_epochs
opt = optax.sgd
for period, info in sch.schedule():
    print(f"{info}")
    print(f"{period} {type(period)}\n")
    sch.transform_optimizer(vstate.parameters, opt, info, period)
    print("\n")

In [ ]:
import flax.linen as nn

arch_name = type(model).__name__
subarch_names = [
    name
    for name in model.__dict__.keys()
    if isinstance(model.__dict__[name], nn.Module)
]
arch_name, subarch_names

In [ ]:
model.__dict__["Trans"][0].__class__.__name__

In [ ]:
def is_subsequence(a, b):
    n, m = len(a), len(b)
    for i in range(m - n + 1):
        if b[i : i + n] == a:
            return True
    return False


a = (1, 2, 3)
b = (1, 2, 4, 3, 4, 5, 6)
is_subsequence(a, b)

In [4]:
factory.setup

frozendict.frozendict({'module': 'Transversal', 'setup': frozendict.frozendict({'Trans_0': frozendict.frozendict({'module': 'Factorized', 'setup': frozendict.frozendict({'site_dependent': True, 'complex': True, 'lattice_size': (4, 4)})}), 'Trans_1': frozendict.frozendict({'module': 'SplitTraining', 'setup': frozendict.frozendict({'modulus_setup': frozendict.frozendict({'module': 'CvT', 'setup': frozendict.frozendict({'n_CP_blocks': (2,), 'channels': (32,), 'attn_heads': (4,), 'kernel': (3, 3), 'final_architecture': (1,), 'lattice_size': (4, 4)})}), 'phase_setup': frozendict.frozendict({'module': 'CNNSzabo', 'setup': frozendict.frozendict({'channels': 40, 'use_bias': True, 'lattice_size': (4, 4)})})})}), 'operation': 'sum', 'symm_Z2': True, 'trivial_Z2': True, 'symm_2D': False, 'irrep': (0, 0), 'use_anchor': False, 'squeeze': True, 'lattice_size': (4, 4)})})